# FX.fact_ohlc — Simple CRUD Test

Insert, read, update, and delete a row from the existing `FX.fact_ohlc` table.

In [3]:
from datetime import datetime, timezone
from decimal import Decimal

from sqlalchemy import text

from imdr.config.settings import get_settings
from imdr.connectors.mssql import MSSQLConnector

connector = MSSQLConnector(get_settings())
print("Connected:", connector.engine.url)

Connected: mssql+pyodbc://@rv-database-1.ctym72ljvrjq.ap-southeast-1.rds.amazonaws.com:1433/IMDR?Trusted_Connection=yes&driver=SQL+Server


## 1. CREATE — Insert a row

In [14]:
insert_sql = text("""
    INSERT INTO FX.fact_ohlc
        (ts, symbol, series, tenor, deal_type, pair_used,
         open_px, high_px, low_px, close_px, mid_px,
         mid_mean_px, mid_median_px, bid, ask, n_ticks)
    OUTPUT INSERTED.id
    VALUES
        (:ts, :symbol, :series, :tenor, :deal_type, :pair_used,
         :open_px, :high_px, :low_px, :close_px, :mid_px,
         :mid_mean_px, :mid_median_px, :bid, :ask, :n_ticks)
""")

params = dict(
    ts=datetime.now(timezone.utc),
    symbol="EURUSD",
    series="spot",
    tenor="ON",
    deal_type="outright",
    pair_used="EURUSD",
    open_px=Decimal("1.08400000"),
    high_px=Decimal("1.08600000"),
    low_px=Decimal("1.08200000"),
    close_px=Decimal("1.08450000"),
    mid_px=Decimal("1.08425000"),
    mid_mean_px=Decimal("1.08410000"),
    mid_median_px=Decimal("1.08420000"),
    bid=Decimal("1.08400000"),
    ask=Decimal("1.08450000"),
    n_ticks=150,
)

with connector.session() as session:
    result = session.execute(insert_sql, params)
    inserted_id = result.scalar()

print(f"Inserted row with id = {inserted_id}")

Inserted row with id = 3


## 2. READ — Fetch the row back

In [4]:
import pandas as pd

read_sql = text("SELECT * FROM FX.fact_ohlc")

with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection())

print(f"Total rows: {len(df)}")
df

Total rows: 34


,id,ts,symbol,series,tenor,deal_type,pair_used,open_px,high_px,low_px,close_px,mid_px,mid_mean_px,mid_median_px,bid,ask,n_ticks,created_at
0,4,2026-03-09 08:00:00 +00:00,USDPHP,NDF_1M,1M,NDF,USDPHP,59.625050,59.692850,59.625000,59.640250,59.640250,59.660872,59.661250,59.634750,59.645750,36000,2026-03-09 09:43:20.2571528
1,5,2026-03-09 08:00:00 +00:00,USDPHP,SPOT,SPOT,NDF,USDPHP,59.455000,59.525500,59.454500,59.473600,59.473600,59.491848,59.493325,59.468600,59.478600,36000,2026-03-09 09:43:20.3961611
2,6,2026-03-09 08:00:00 +00:00,USDTWD,NDF_1M,1M,NDF,USDTWD,31.930150,31.968675,31.929400,31.940375,31.940375,31.947321,31.947525,31.939425,31.941325,36000,2026-03-09 09:43:20.4101641
3,7,2026-03-09 08:00:00 +00:00,USDTWD,SPOT,SPOT,NDF,USDTWD,31.854250,31.888325,31.850625,31.859750,31.859750,31.867133,31.866500,31.858800,31.860700,36000,2026-03-09 09:43:20.4241498
4,8,2026-03-09 08:00:00 +00:00,USDIDR,NDF_1M,1M,NDF,USDIDR,16969.275000,17003.990085,16966.687500,16985.990000,16985.990000,16986.743686,16990.172500,16984.490000,16987.490000,36000,2026-03-09 09:43:20.4391541
5,9,2026-03-09 08:00:00 +00:00,USDIDR,SPOT,SPOT,NDF,USDIDR,16945.802500,16981.900000,16943.075000,16961.990000,16961.990000,16962.728847,16965.825000,16960.490000,16963.490000,36000,2026-03-09 09:43:20.4551504
6,10,2026-03-09 08:00:00 +00:00,USDTHB,NDF_1M,1M,NDF,USDTHB,32.057334,32.156683,32.049333,32.123033,32.123033,32.109918,32.122833,32.111033,32.135033,36000,2026-03-09 09:43:20.4701484
7,11,2026-03-09 08:00:00 +00:00,USDTHB,SPOT,SPOT,NDF,USDTHB,32.057617,32.155800,32.047650,32.118500,32.118500,32.108561,32.119700,32.114700,32.122300,36000,2026-03-09 09:43:20.4851489
8,12,2026-03-09 08:00:00 +00:00,USDINR,NDF_1M,1M,NDF,USDINR,92.678250,92.775550,92.675700,92.756675,92.756675,92.744329,92.748685,92.755175,92.758175,36000,2026-03-09 09:43:20.5001470
9,13,2026-03-09 08:00:00 +00:00,USDINR,SPOT,SPOT,NDF,USDINR,92.251950,92.341000,92.250475,92.318450,92.318450,92.306712,92.309750,92.316950,92.319950,36000,2026-03-09 09:43:20.5141489


## 3. UPDATE — Change close_px

In [16]:
update_sql = text("""
    UPDATE FX.fact_ohlc
    SET close_px = :close_px
    WHERE id = :id
""")

with connector.session() as session:
    session.execute(update_sql, {"close_px": Decimal("1.09000000"), "id": inserted_id})

# Verify the update
with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Updated close_px = {df['close_px'].iloc[0]}")
df

Updated close_px = 1.09


,id,ts,symbol,series,tenor,deal_type,pair_used,open_px,high_px,low_px,close_px,mid_px,mid_mean_px,mid_median_px,bid,ask,n_ticks,created_at
0,3,2026-03-09 09:25:05 +00:00,EURUSD,spot,ON,outright,EURUSD,1.084,1.086,1.082,1.09,1.08425,1.0841,1.0842,1.084,1.0845,150,2026-03-09 09:25:05.3396233


## 4. DELETE — Remove the test row

In [17]:
delete_sql = text("DELETE FROM FX.fact_ohlc WHERE id = :id")

with connector.session() as session:
    result = session.execute(delete_sql, {"id": inserted_id})

print(f"Deleted {result.rowcount} row(s)")

# Verify it's gone
with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Rows remaining with id={inserted_id}: {len(df)}")

Deleted 1 row(s)
Rows remaining with id=3: 0


In [18]:
connector.dispose()
print("Done — connection pool closed.")

Done — connection pool closed.
